# "Hello World" example with Prefect+Dask

See the associated:

  * Python module: [hello_world_flow.py](./hello_world_flow.py)
  * YAML file: [hello_world_flow.yaml](./hello_world_flow.yaml)

## Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/41e8ad0cce974322aec607c3f3cbe240/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/2
Dask workers for 'dask-eopf' are up: 2/2


In [3]:
# Other imports
import os
from importlib import reload
from rs_common import prefect_utils
from rs_common.prefect_utils import *

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

In [4]:
# We use only the eopf dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [5]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [6]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./hello_world_flow.yaml"

10:24:39.549 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.war

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'hello-world/sprint19-hello-world' successfully created with id   │
│ 'f2b000a6-c9f9-456f-9ce0-c4aa03fe1e31'.                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/f2b000a6-c9f9-456f-9ce0-c4aa03fe1e31


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'hello-world/sprint19-hello-world'



In [7]:
deploy_name = "hello-world/sprint19-hello-world"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'hello-world/sprint19-hello-world'


## Run Prefect flow

In [8]:
%%bash -s "$deploy_name"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --watch

Creating flow run for deployment 'hello-world/sprint19-hello-world'...
Created flow run 'illegal-fossa'.
└── UUID: d37dd7e6-bcae-4a53-a09b-4267a0e34207
└── Parameters: {}
└── Job Variables: {}
└── Scheduled start time: 2025-05-22 10:24:45 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/d37dd7e6-bcae-4a53-a09b-4267a0e34207
Watching flow run 'illegal-fossa'...


10:24:47.728 | INFO    | prefect - Flow run is in state 'Pending'
10:24:51.820 | INFO    | prefect - Flow run is in state 'Running'
10:24:54.313 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


NOTE: we could also call the Prefect flow from Python code. This is useful to debug.

In [9]:
run_from_python = False
if run_from_python:
    # Import the module, or reload it if you changed its source code
    import hello_world_flow
    reload(hello_world_flow)
    
    # Run the flow
    results = hello_world_flow.hello_world()
    display(results)

## 3. Shutdown the dask clusters

In [10]:
shutdown = False
if shutdown:
    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.